# Lab Activity 5: CNN Lab (CIFAR-10)
**Amrita School of Computing, Chennai**

This notebook implements the CNN described in Part 1 of the worksheet, trains it on the CIFAR-10 dataset, and contains worked code for Part 2, Part 3 and Part 4 of the activity sheet.

In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cpu


## Part 1: Model Definition

CNN architecture:
- Input: 32x32x3 (RGB image, CIFAR-10)
- 2 Convolutional layers: 3x3 kernels, 'same' padding
- 2 Pooling layers: 2x2 Max Pooling, stride 2
- 2 Dense layers: one hidden layer + one output layer (10 classes)

In [2]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        # Conv Layer 1: 3x3 kernel, 'same' padding -> padding=1, 32 filters
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Conv Layer 2: 3x3 kernel, 'same' padding -> padding=1, 64 filters
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # After two 2x2/stride-2 pools: 32 -> 16 -> 8, so feature map is 8x8 with 64 channels
        self.flatten_dim = 64 * 8 * 8

        # Dense layers: one hidden layer + one output layer (10 classes)
        self.fc1 = nn.Linear(self.flatten_dim, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))   # (B, 32, 16, 16)
        x = self.pool2(F.relu(self.conv2(x)))   # (B, 64, 8, 8)
        x = torch.flatten(x, 1)                 # (B, 4096)
        x = F.relu(self.fc1(x))                 # (B, 128)
        x = self.fc2(x)                          # (B, 10)
        return x

model = SimpleCNN().to(device)
print(model)

SimpleCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=4096, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)


## Part 2: Dimension Tracking

Run a dummy batch through the model layer-by-layer and print the tensor shape after every operation (Batch size = 32).

In [3]:
x = torch.randn(32, 3, 32, 32).to(device)
print('Input Image:            ', tuple(x.shape))

x1 = F.relu(model.conv1(x))
print('Conv Layer 1 output:    ', tuple(x1.shape))

x1p = model.pool1(x1)
print('Max Pool 1 output:      ', tuple(x1p.shape))

x2 = F.relu(model.conv2(x1p))
print('Conv Layer 2 output:    ', tuple(x2.shape))

x2p = model.pool2(x2)
print('Max Pool 2 output:      ', tuple(x2p.shape))

xf = torch.flatten(x2p, 1)
print('Flatten Layer output:   ', tuple(xf.shape))

Input Image:             (32, 3, 32, 32)
Conv Layer 1 output:     (32, 32, 32, 32)
Max Pool 1 output:       (32, 32, 16, 16)
Conv Layer 2 output:     (32, 64, 16, 16)
Max Pool 2 output:       (32, 64, 8, 8)
Flatten Layer output:    (32, 4096)


## Load CIFAR-10 dataset (from local path, no download) and train the model

In [4]:
# Your batch files (batches.meta, data_batch_1..5, test_batch) sit directly in
# /content/sample_data -- there is no 'cifar-10-batches-py' wrapper folder, so
# torchvision.datasets.CIFAR10 (which requires that wrapper) won't find them.
# This custom Dataset reads the pickled batch files directly from that path.
import os
import pickle
import numpy as np
from PIL import Image
from torch.utils.data import Dataset

data_root = '/content/sample_data'

class CIFAR10Local(Dataset):
    def __init__(self, root, train=True, transform=None):
        self.root = root
        self.transform = transform
        if train:
            batch_files = [f'data_batch_{i}' for i in range(1, 6)]
        else:
            batch_files = ['test_batch']

        images, labels = [], []
        for bf in batch_files:
            path = os.path.join(root, bf)
            with open(path, 'rb') as f:
                entry = pickle.load(f, encoding='latin1')
            images.append(entry['data'])
            labels.extend(entry['labels'] if 'labels' in entry else entry['fine_labels'])

        self.data = np.vstack(images).reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)  # NHWC, uint8
        self.targets = labels

        meta_path = os.path.join(root, 'batches.meta')
        with open(meta_path, 'rb') as f:
            meta = pickle.load(f, encoding='latin1')
        self.classes = meta['label_names']

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img = Image.fromarray(self.data[idx])
        label = self.targets[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

# Data augmentation (see Part 3, Q2) + normalization
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

train_set = CIFAR10Local(root=data_root, train=True, transform=transform_train)
test_set = CIFAR10Local(root=data_root, train=False, transform=transform_test)

train_loader = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False, num_workers=2)

classes = train_set.classes
print('Train size:', len(train_set), '| Test size:', len(test_set), '| Classes:', classes)

Train size: 50000 | Test size: 10000 | Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [5]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f'Epoch {epoch+1}/{num_epochs}  Loss: {running_loss/len(train_loader):.4f}')

Epoch 1/10  Loss: 1.4732
Epoch 2/10  Loss: 1.1434
Epoch 3/10  Loss: 1.0144
Epoch 4/10  Loss: 0.9382
Epoch 5/10  Loss: 0.8889
Epoch 6/10  Loss: 0.8558
Epoch 7/10  Loss: 0.8148
Epoch 8/10  Loss: 0.7973
Epoch 9/10  Loss: 0.7708
Epoch 10/10  Loss: 0.7550


In [6]:
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Test Accuracy: {100 * correct / total:.2f}%')

Test Accuracy: 74.99%


## Part 3: CNN-Specific Logic

### Debugging Challenge

The buggy snippet (grayscale 28x28x1 input):
```python
self.conv1 = nn.Conv2d(1, 10, kernel_size=5, stride=1, padding=0)  # -> 24x24x10
self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)                 # -> 12x12x10
self.fc1 = nn.Linear(in_features=28*28, out_features=50)           # BUG
```

The demonstration below shows exactly where the shape mismatch occurs and the fix.

In [7]:
class BuggyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 10, kernel_size=5, stride=1, padding=0)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(in_features=28*28, out_features=50)
    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = torch.flatten(x, 1)
        print('Flattened shape entering fc1:', x.shape)
        x = self.fc1(x)
        return x

buggy = BuggyNet()
dummy = torch.randn(4, 1, 28, 28)
try:
    buggy(dummy)
except RuntimeError as e:
    print('RuntimeError raised at fc1:', e)

Flattened shape entering fc1: torch.Size([4, 1440])
RuntimeError raised at fc1: mat1 and mat2 shapes cannot be multiplied (4x1440 and 784x50)


In [8]:
H_after_conv1 = (28 - 5) // 1 + 1
H_after_pool1 = H_after_conv1 // 2
correct_in_features = H_after_pool1 * H_after_pool1 * 10
print('Correct in_features for fc1:', correct_in_features)

class FixedNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 10, kernel_size=5, stride=1, padding=0)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(in_features=correct_in_features, out_features=50)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        return x

fixed = FixedNet()
print(fixed(dummy).shape)

Correct in_features for fc1: 1440
torch.Size([4, 50])


## Part 4: Implementation Exercises

### 1. Parameter Calculation Code

In [9]:
def count_params(kernel_size, in_channels, out_channels):
    """Total trainable parameters (weights + biases) for a standard conv layer."""
    weights = kernel_size * kernel_size * in_channels * out_channels
    biases = out_channels
    return weights + biases

print(count_params(3, 3, 32))
print(sum(p.numel() for p in model.conv1.parameters()))

896
896


### 2. Custom Pooling Logic

Given 4x4 input matrix:
```
[12,  20,  30,   0]
[ 8,  12,   2,   0]
[34,  70,  37,   4]
[112, 100, 25,  12]
```

Pseudocode:
```
for i in range(0, H, stride):
    for j in range(0, W, stride):
        window = input[i : i+pool_size, j : j+pool_size]
        output[i/stride][j/stride] = max(window)
```

In [10]:
def max_pool_2x2_stride2(matrix):
    H, W = len(matrix), len(matrix[0])
    out_h, out_w = H // 2, W // 2
    output = [[0] * out_w for _ in range(out_h)]
    for i in range(0, H, 2):
        for j in range(0, W, 2):
            window = [matrix[i][j], matrix[i][j+1], matrix[i+1][j], matrix[i+1][j+1]]
            output[i // 2][j // 2] = max(window)
    return output

input_matrix = [
    [12,  20,  30,  0],
    [ 8,  12,   2,  0],
    [34,  70,  37,  4],
    [112, 100, 25, 12],
]

print(max_pool_2x2_stride2(input_matrix))

[[20, 30], [112, 37]]


### 3. Layer Output Logic

In [11]:
def conv_output_size(W, K, P, S):
    """Output spatial size of a convolutional layer.
    W: input width, K: kernel size, P: padding, S: stride."""
    return math.floor((W - K + 2 * P) / S) + 1

print(conv_output_size(28, 5, 0, 1))

print(conv_output_size(32, 3, 1, 1))

24
32
